In [ ]:
import gc
import os
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from xgboost import XGBRegressor
    xgboost_is_available = True
except Exception:
    xgboost_is_available = False
    print("XGBoost is not installed. XGBoost runs will be skipped.")

# --------------------------------------------------
# Config
# --------------------------------------------------
GOLD_PATH = r"..\..\..\Data\c_final_tables\baseline_model_gold_table\I66_phase1_TTI_features_2022_2025_combined.parquet"
OUTPUT_DIRECTORY = r"..\..\..\notebooks\modelling\project_1_output\combined_horizon_models_2022_2025"

os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)

gold_dataframe = pd.read_parquet(GOLD_PATH)
gold_dataframe = gold_dataframe.sort_values(["tmc", "direction", "ts_utc"]).reset_index(drop=True)
gold_dataframe = gold_dataframe[gold_dataframe["ts_local"].dt.year <= 2025]

print(gold_dataframe.shape)
gold_dataframe.head()


(17229104, 44)


,tmc,direction,ts_utc,ts_local,road,miles,tti,confidence,tti_missing_flag,year,...,tti_lag_11,tti_lag_12,tti_rolling_mean_6,tti_rolling_mean_12,tti_rolling_standard_deviation_6,tti_rolling_standard_deviation_12,tti_change_1_step,tti_absolute_change_1_step,is_low_confidence_flag,source_year
0,110+04164,WESTBOUND,2022-01-01 06:00:00+00:00,2022-01-01 01:00:00-05:00,I-66,0.103042,0.892857,30.0,0,2022.0,...,0.909091,0.960000,0.947991,0.946425,0.066195,0.047097,-0.004786,0.004786,0,2022
1,110+04164,WESTBOUND,2022-01-01 06:05:00+00:00,2022-01-01 01:05:00-05:00,I-66,0.103042,0.944882,30.0,0,2022.0,...,0.949179,0.909091,0.930687,0.940829,0.064470,0.049276,0.020130,0.020130,0,2022
2,110+04164,WESTBOUND,2022-01-01 06:10:00+00:00,2022-01-01 01:10:00-05:00,I-66,0.103042,0.934034,30.0,0,2022.0,...,0.926641,0.949179,0.917955,0.943812,0.048588,0.048252,0.052025,0.052025,0,2022
3,110+04164,WESTBOUND,2022-01-01 06:15:00+00:00,2022-01-01 01:15:00-05:00,I-66,0.103042,0.933852,30.0,0,2022.0,...,0.960384,0.926641,0.906926,0.942550,0.030223,0.048297,-0.010848,0.010848,0,2022
4,110+04164,WESTBOUND,2022-01-01 06:20:00+00:00,2022-01-01 01:20:00-05:00,I-66,0.103042,0.923077,30.0,0,2022.0,...,0.963855,0.960384,0.909311,0.943151,0.031934,0.048126,-0.000182,0.000182,0,2022


In [ ]:

# --------------------------------------------------
# Time split
# Main paper recommendation:
# Train = 2022-2024
# Validation = 2025-01-01 to 2025-06-30
# Test = 2025-07-01 onward
# --------------------------------------------------
train_end_date = "2024-12-31 23:59:59+00:00"
validation_end_date = "2025-06-30 23:59:59+00:00"

# --------------------------------------------------
# Feature columns
# --------------------------------------------------
feature_columns = [
    col for col in gold_dataframe.columns
    if col.startswith("tti_lag_")
    or col.startswith("tti_rolling_")
    or col in [
        "tti_change_1_step",
        "tti_absolute_change_1_step",
        "is_low_confidence_flag",
        "sin_hour",
        "cos_hour",
        "sin_day_of_week",
        "cos_day_of_week",
        "sin_month",
        "cos_month",
        "is_weekend_flag",
    ]
]

feature_columns = [col for col in feature_columns if col in gold_dataframe.columns]

horizon_target_map = {
    "5min": "target_tti_5min_ahead",
    "15min": "target_tti_15min_ahead",
    "30min": "target_tti_30min_ahead",
}

print("Number of feature columns:", len(feature_columns))
print(feature_columns)


Number of feature columns: 26
['is_weekend_flag', 'sin_hour', 'cos_hour', 'sin_day_of_week', 'cos_day_of_week', 'sin_month', 'cos_month', 'tti_lag_1', 'tti_lag_2', 'tti_lag_3', 'tti_lag_4', 'tti_lag_5', 'tti_lag_6', 'tti_lag_7', 'tti_lag_8', 'tti_lag_9', 'tti_lag_10', 'tti_lag_11', 'tti_lag_12', 'tti_rolling_mean_6', 'tti_rolling_mean_12', 'tti_rolling_standard_deviation_6', 'tti_rolling_standard_deviation_12', 'tti_change_1_step', 'tti_absolute_change_1_step', 'is_low_confidence_flag']


In [ ]:

def calculate_regression_metrics(true_values, predicted_values):
    mae_value = mean_absolute_error(true_values, predicted_values)
    rmse_value = np.sqrt(mean_squared_error(true_values, predicted_values))
    return {
        "mae": float(mae_value),
        "rmse": float(rmse_value),
    }

def create_time_split_masks(dataframe):
    training_mask = dataframe["ts_utc"] <= train_end_date
    validation_mask = (
        (dataframe["ts_utc"] > train_end_date)
        & (dataframe["ts_utc"] <= validation_end_date)
    )
    testing_mask = dataframe["ts_utc"] > validation_end_date
    return training_mask, validation_mask, testing_mask

def create_time_split(dataframe):
    training_dataframe = dataframe[dataframe["ts_utc"] <= train_end_date].copy()
    validation_dataframe = dataframe[
        (dataframe["ts_utc"] > train_end_date)
        & (dataframe["ts_utc"] <= validation_end_date)
    ].copy()
    testing_dataframe = dataframe[dataframe["ts_utc"] > validation_end_date].copy()
    return training_dataframe, validation_dataframe, testing_dataframe


def fit_linear_regression_model(training_features, training_target):
    model = LinearRegression()
    model.fit(training_features, training_target)
    return model


def fit_random_forest_model(training_features, training_target):

    sample_size = min(500000, len(training_features))
    rng = np.random.default_rng(42)

    sample_index = rng.choice(
        len(training_features),
        size=sample_size,
        replace=False
    )

    X_sample = training_features[sample_index]
    y_sample = training_target[sample_index]

    model = RandomForestRegressor(
        n_estimators=50,
        max_depth=12,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=1,
    )

    model.fit(X_sample, y_sample)

    return model


def fit_xgboost_model(training_features, training_target):
    if not xgboost_is_available:
        return None

    model = XGBRegressor(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
    )
    model.fit(training_features, training_target)
    return model
def train_model(model_name, training_features, training_target):
    if model_name == "Persistence":
        return None

    if model_name == "LinearRegression":
        model = fit_linear_regression_model(training_features, training_target)
    elif model_name == "RandomForest":
        model = fit_random_forest_model(training_features, training_target)
    elif model_name == "XGBoost":
        model = fit_xgboost_model(training_features, training_target)
        if model is None:
            return None
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    return model

def generate_predictions_from_trained_model(model_name, trained_model, evaluation_features, persistence_values=None):
    if model_name == "Persistence":
        return persistence_values

    return trained_model.predict(evaluation_features)

# def generate_predictions(model_name, training_features, training_target, evaluation_dataframe):
#     if model_name == "Persistence":
#         return evaluation_dataframe["tti"].to_numpy()

#     if model_name == "LinearRegression":
#         model = fit_linear_regression_model(training_features, training_target)
#     elif model_name == "RandomForest":
#         model = fit_random_forest_model(training_features, training_target)
#     elif model_name == "XGBoost":
#         model = fit_xgboost_model(training_features, training_target)
#         if model is None:
#             return None
#     else:
#         raise ValueError(f"Unsupported model: {model_name}")

#     return model.predict(evaluation_dataframe[feature_columns])


In [ ]:

# --------------------------------------------------
# Run combined horizon-model experiment
# Exports:
# 1. overall_horizon_model_results.csv
# 2. per_year_horizon_model_results.csv
# 3. per_tmc_horizon_model_results.csv
# --------------------------------------------------
overall_results = []
per_year_results = []
per_tmc_results = []

model_names = ["Persistence", "LinearRegression", "RandomForest", "XGBoost"]

for horizon_name, target_column in horizon_target_map.items():
    horizon_dataframe = gold_dataframe.dropna(subset=[target_column] + feature_columns)

    selected_columns = ["ts_utc", "year", "tmc", "direction", "tti", target_column] + feature_columns
    selected_columns = [col for col in selected_columns if col in horizon_dataframe.columns]

    horizon_dataframe = horizon_dataframe[selected_columns]

    training_mask, validation_mask, testing_mask = create_time_split_masks(horizon_dataframe)

    training_dataframe = horizon_dataframe.loc[training_mask, :]
    validation_dataframe = horizon_dataframe.loc[validation_mask, :]
    testing_dataframe = horizon_dataframe.loc[testing_mask, :]

    # training_dataframe, validation_dataframe, testing_dataframe = create_time_split(horizon_dataframe)

    print("--------------------------------------------------")
    print("Horizon:", horizon_name)
    print("Training rows:", len(training_dataframe))
    print("Validation rows:", len(validation_dataframe))
    print("Testing rows:", len(testing_dataframe))

    X_train = training_dataframe[feature_columns].to_numpy(dtype=np.float32, copy=True)
    y_train = training_dataframe[target_column].to_numpy(dtype=np.float32, copy=True)

    X_validation = validation_dataframe[feature_columns].to_numpy(dtype=np.float32, copy=True)
    y_validation = validation_dataframe[target_column].to_numpy(dtype=np.float32, copy=True)

    X_test = testing_dataframe[feature_columns].to_numpy(dtype=np.float32, copy=True)
    y_test = testing_dataframe[target_column].to_numpy(dtype=np.float32, copy=True)

    for model_name in model_names:
        print(f"Currently testing {model_name} model")
        if model_name == "XGBoost" and not xgboost_is_available:
            continue

        trained_model = train_model(
            model_name,
            X_train,
            y_train,
        )

        if model_name != "Persistence" and trained_model is None:
            continue

        validation_persistence = validation_dataframe["tti"].to_numpy(dtype=np.float32, copy=True)
        testing_persistence = testing_dataframe["tti"].to_numpy(dtype=np.float32, copy=True)

        validation_predictions = generate_predictions_from_trained_model(
            model_name,
            trained_model,
            X_validation,
            validation_persistence,
        )

        testing_predictions = generate_predictions_from_trained_model(
            model_name,
            trained_model,
            X_test,
            testing_persistence,
        )

        validation_metrics = calculate_regression_metrics(
            validation_dataframe[target_column],
            validation_predictions,
        )
        testing_metrics = calculate_regression_metrics(
            testing_dataframe[target_column],
            testing_predictions,
        )

        overall_results.append({
            "horizon": horizon_name,
            "model": model_name,
            "train_rows": len(training_dataframe),
            "validation_rows": len(validation_dataframe),
            "test_rows": len(testing_dataframe),
            "validation_mae": validation_metrics["mae"],
            "validation_rmse": validation_metrics["rmse"],
            "test_mae": testing_metrics["mae"],
            "test_rmse": testing_metrics["rmse"],
        })

        validation_with_predictions = validation_dataframe[["tmc", "direction", "year", target_column]].copy()
        validation_with_predictions["prediction"] = validation_predictions
        validation_with_predictions["absolute_error"] = (
            validation_with_predictions[target_column] - validation_with_predictions["prediction"]
        ).abs()
        validation_with_predictions["squared_error"] = (
            validation_with_predictions[target_column] - validation_with_predictions["prediction"]
        ) ** 2

        testing_with_predictions = testing_dataframe[["tmc", "direction", "year", target_column]].copy()
        testing_with_predictions["prediction"] = testing_predictions
        testing_with_predictions["absolute_error"] = (
            testing_with_predictions[target_column] - testing_with_predictions["prediction"]
        ).abs()
        testing_with_predictions["squared_error"] = (
            testing_with_predictions[target_column] - testing_with_predictions["prediction"]
        ) ** 2

        validation_year_summary = (
            validation_with_predictions
            .groupby("year")
            .agg(
                row_count=(target_column, "size"),
                mae=("absolute_error", "mean"),
                mse=("squared_error", "mean"),
            )
            .reset_index()
        )
        validation_year_summary["rmse"] = np.sqrt(validation_year_summary["mse"])
        validation_year_summary["split"] = "validation"
        validation_year_summary["horizon"] = horizon_name
        validation_year_summary["model"] = model_name

        testing_year_summary = (
            testing_with_predictions
            .groupby("year")
            .agg(
                row_count=(target_column, "size"),
                mae=("absolute_error", "mean"),
                mse=("squared_error", "mean"),
            )
            .reset_index()
        )
        testing_year_summary["rmse"] = np.sqrt(testing_year_summary["mse"])
        testing_year_summary["split"] = "test"
        testing_year_summary["horizon"] = horizon_name
        testing_year_summary["model"] = model_name

        per_year_results.extend(validation_year_summary[["year", "split", "horizon", "model", "row_count", "mae", "rmse"]].to_dict("records"))
        per_year_results.extend(testing_year_summary[["year", "split", "horizon", "model", "row_count", "mae", "rmse"]].to_dict("records"))

        validation_tmc_summary = (
            validation_with_predictions
            .groupby(["tmc", "direction"])
            .agg(
                row_count=(target_column, "size"),
                mae=("absolute_error", "mean"),
                mse=("squared_error", "mean"),
            )
            .reset_index()
        )
        validation_tmc_summary["rmse"] = np.sqrt(validation_tmc_summary["mse"])
        validation_tmc_summary["split"] = "validation"
        validation_tmc_summary["horizon"] = horizon_name
        validation_tmc_summary["model"] = model_name

        testing_tmc_summary = (
            testing_with_predictions
            .groupby(["tmc", "direction"])
            .agg(
                row_count=(target_column, "size"),
                mae=("absolute_error", "mean"),
                mse=("squared_error", "mean"),
            )
            .reset_index()
        )
        testing_tmc_summary["rmse"] = np.sqrt(testing_tmc_summary["mse"])
        testing_tmc_summary["split"] = "test"
        testing_tmc_summary["horizon"] = horizon_name
        testing_tmc_summary["model"] = model_name

        per_tmc_results.extend(validation_tmc_summary[["tmc", "direction", "split", "horizon", "model", "row_count", "mae", "rmse"]].to_dict("records"))
        per_tmc_results.extend(testing_tmc_summary[["tmc", "direction", "split", "horizon", "model", "row_count", "mae", "rmse"]].to_dict("records"))

    # cleanup after finishing all models for this horizon
    del training_dataframe, validation_dataframe, testing_dataframe, horizon_dataframe
    import gc
    gc.collect()

overall_results_dataframe = pd.DataFrame(overall_results)
per_year_results_dataframe = pd.DataFrame(per_year_results)
per_tmc_results_dataframe = pd.DataFrame(per_tmc_results)

overall_results_path = os.path.join(OUTPUT_DIRECTORY, "overall_horizon_model_results.csv")
per_year_results_path = os.path.join(OUTPUT_DIRECTORY, "per_year_horizon_model_results.csv")
per_tmc_results_path = os.path.join(OUTPUT_DIRECTORY, "per_tmc_horizon_model_results.csv")

overall_results_dataframe.to_csv(overall_results_path, index=False)
per_year_results_dataframe.to_csv(per_year_results_path, index=False)
per_tmc_results_dataframe.to_csv(per_tmc_results_path, index=False)

print("Saved:")
print(overall_results_path)
print(per_year_results_path)
print(per_tmc_results_path)


--------------------------------------------------
Horizon: 5min
Training rows: 12920705
Validation rows: 2136489
Testing rows: 2171910
Currently testing Persistence model
Currently testing LinearRegression model
Currently testing RandomForest model
Currently testing XGBoost model
--------------------------------------------------
Horizon: 15min
Training rows: 12920705
Validation rows: 2136489
Testing rows: 2171910
Currently testing Persistence model
Currently testing LinearRegression model
Currently testing RandomForest model
Currently testing XGBoost model
--------------------------------------------------
Horizon: 30min
Training rows: 12920705
Validation rows: 2136489
Testing rows: 2171910
Currently testing Persistence model
Currently testing LinearRegression model
Currently testing RandomForest model
Currently testing XGBoost model
Saved:
..\..\..\notebooks\modelling\model_outputs\Phase_1_model\combined_horizon_models_2022_2025\overall_horizon_model_results.csv
..\..\..\notebooks\m

In [ ]:

overall_results_dataframe.sort_values(["horizon", "test_mae"]).reset_index(drop=True)


,horizon,model,train_rows,validation_rows,test_rows,validation_mae,validation_rmse,test_mae,test_rmse
0,15min,Persistence,12920705,2136489,2171910,0.118281,0.372073,0.113659,0.364218
1,15min,XGBoost,12920705,2136489,2171910,0.122721,0.370699,0.118391,0.364543
2,15min,RandomForest,12920705,2136489,2171910,0.122495,0.370806,0.118595,0.366038
3,15min,LinearRegression,12920705,2136489,2171910,0.132274,0.394077,0.128724,0.387305
4,30min,RandomForest,12920705,2136489,2171910,0.147393,0.432638,0.143261,0.428885
5,30min,XGBoost,12920705,2136489,2171910,0.148357,0.432989,0.143443,0.427114
6,30min,Persistence,12920705,2136489,2171910,0.156121,0.489694,0.149337,0.477489
7,30min,LinearRegression,12920705,2136489,2171910,0.163929,0.471023,0.160476,0.463761
8,5min,Persistence,12920705,2136489,2171910,0.072154,0.218488,0.069605,0.214531
9,5min,RandomForest,12920705,2136489,2171910,0.093091,0.287483,0.089925,0.282101


In [ ]:

# Helpful paper-ready summary table using test metrics only
paper_summary_table = (
    overall_results_dataframe[
        ["horizon", "model", "test_mae", "test_rmse"]
    ]
    .sort_values(["horizon", "test_mae"])
    .reset_index(drop=True)
)

paper_summary_table_path = os.path.join(OUTPUT_DIRECTORY, "paper_summary_table.csv")
paper_summary_table.to_csv(paper_summary_table_path, index=False)

print(paper_summary_table)
print("Saved:", paper_summary_table_path)


   horizon             model  test_mae  test_rmse
0    15min       Persistence  0.113659   0.364218
1    15min           XGBoost  0.118391   0.364543
2    15min      RandomForest  0.118595   0.366038
3    15min  LinearRegression  0.128724   0.387305
4    30min      RandomForest  0.143261   0.428885
5    30min           XGBoost  0.143443   0.427114
6    30min       Persistence  0.149337   0.477489
7    30min  LinearRegression  0.160476   0.463761
8     5min       Persistence  0.069605   0.214531
9     5min      RandomForest  0.089925   0.282101
10    5min           XGBoost  0.090622   0.286604
11    5min  LinearRegression  0.096341   0.296565
Saved: ..\..\..\notebooks\modelling\model_outputs\Phase_1_model\combined_horizon_models_2022_2025\paper_summary_table.csv
